In [1]:
import sys
sys.path.insert(1, '/home/santolam/MyPythonLibrary/MAR_HMA_evaluation_paper/libmar/')
import climbasis as climb
from climbasis import *
import domain as dom
import numpy.ma as ma



In [2]:
domain='GRi';spin='spin2';var='T';units='°C';domainM='GRp'

nx=10;ny=10


In [3]:
sourceData=f'/bettik/PROJECTS/pr-regional-climate/santolam/MARout_post/{domainM}/spin2/work/monthly/'


In [4]:
ds_grM= xr.open_dataset('/home/santolam/NST.2014.01.01.00.GRi.nc')
LONS=ds_grM.LON[ny:-ny,nx:-nx].values
LATS=ds_grM.LAT[ny:-ny,nx:-nx].values
dsSH=ds_grM.SH[ny:-ny,nx:-nx]
dsICE=ds_grM.ICE[ny:-ny,nx:-nx]
latS=LATS.min(); latN=LATS.max()
lonW=LONS.min(); lonE=LONS.max()

field0_SH=dsSH;
field0_ICE=dsICE;

In [5]:
#!ls /bettik/PROJECTS/pr-regional-climate/santolam/MARout/GRp/spin2/work/monthly/TT*

In [6]:
iyr=1960
fyr=2020

##ax0
ds0_raw= xr.open_dataset(sourceData+
                           'TTZ_mon_MARv3.14_ER5_spin2_'+domainM+'_1950-2025.nc')['TTZ'][:,0,:,:] ## montlhy mean
ds0 = ds0_raw.rename({'TIME': 'time'})
field0=ds0[:,ny:-ny,nx:-nx].sel(time=slice(str(iyr),str(fyr)))

annual_mean_f0 = field0.resample(time='YS').mean()
clim_f0 = annual_mean_f0.mean(dim='time')


In [7]:
#print(ds0.time)

In [8]:
##ax1

ds1_raw = xr.open_dataset('/bettik/PROJECTS/pr-regional-climate/santolam/cru/'
                      +'tmp_cru405_mon_19010116-20201216_05x05_GLOB.nc')['tmp']

field1 =ds1_raw.sel(time=slice(str(iyr),str(fyr))).sel(lat=slice(latS,latN),lon=slice(lonW,lonE))
annual_mean_f1 = field1.resample(time='YS').mean()
clim_f1 = annual_mean_f1.mean(dim='time')

ds1_SH=xr.open_dataset('/bettik/PROJECTS/pr-regional-climate/santolam/gmted/'
                       +'gmted_on_cru_hma_grid.nc')['elevation']
#ds5_SH=dom.reversing_lat(ds5_SH_raw)
field1_SH=ds1_SH.sel(lat=slice(latS,latN),lon=slice(lonW,lonE))


In [9]:
#clim_f1.plot()

In [10]:
#clim_f1.max(),clim_f1.min()

In [ ]:
##ax2
ds2_raw=xr.open_dataset(f'/bettik/PROJECTS/pr-regional-climate/santolam/MARout_coarser/'+
                            f'TTZ_mon_MARv3.14_ER5_spin2_{domainM}_1950-2025_cru05x05.nc')['TTZ'][:,0,:,:]
mask = np.isfinite(ds2_raw).any(dim='TIME')  # problem of global, all grid nan
ds2_raw_mask= ds2_raw.where(mask, drop=True)

ds2= ds2_raw_mask.rename({'TIME': 'time','x':'longitude','y':'latitude'})

field2=ds2.sel(time=slice(str(iyr), str(fyr))).sel(latitude=slice(latS,latN),longitude=slice(lonW,lonE))

annual_mean_f2 = field2.resample(time='YS').mean()
clim_f2= annual_mean_f2.mean(dim='time')

bias_f2=clim_f2.values-clim_f1.values


In [ ]:
###ax3
iyr=1960
fyr=2015
ds_RF_raw=xr.open_dataset(f'/bettik/PROJECTS/pr-regional-climate/santolam/MARout_post/{domainM}/spin2/work/monthly/'
                          +'RF_mon_MARv3.14_ER5_spin2_'+domainM+'_1950-2025.nc')['RF']
ds_RF = ds_RF_raw.rename({'TIME': 'time'})
###
ds_SF_raw=xr.open_dataset(f'/bettik/PROJECTS/pr-regional-climate/santolam/MARout_post/{domainM}/spin2/work/monthly/'
                        +'SF_mon_MARv3.14_ER5_spin2_'+domainM+'_1950-2025.nc')['SF']
ds_SF = ds_SF_raw.rename({'TIME': 'time'})

ds3=ds_RF+ds_SF

field3=ds3[:,ny:-ny,nx:-nx].sel(time=slice(str(iyr), str(fyr)))

annual_mean_f3 = field3.resample(time='YS').mean()
clim_f3 = annual_mean_f3.mean(dim='time')



In [ ]:
#clim_f3.plot()

In [ ]:
#print(clim_f3.max(),clim_f3.min())

In [ ]:
#ax4
ds4 = xr.open_dataset('/bettik/PROJECTS/pr-regional-climate/santolam/aphro/monthly/'
                      +'APHRO_mon_MA_025deg_V1101_EXR1.1951-2015.nc',decode_times=False)['precip']
start_date = pd.Timestamp('1951-01-01')
# Create new time index
nmonths = ds4.time.size
time_new = pd.date_range(start=start_date, periods=nmonths, freq='MS')  # 'MS' = Month Start
# Replace time in dataset
ds4['time'] = time_new
ds4=dom.reversing_lat(ds4)
ds4=ds4.rename({'latitude':'lat','longitude':'lon'})
field4_mm_day=ds4.sel(lat=slice(latN,latS),lon=slice(lonW,lonE))
field4_mm_day=field4_mm_day.sel(time=slice(str(iyr),str(fyr)))

# Convert mm/day to mm/month
days_in_month = field4_mm_day['time'].dt.days_in_month
field4= field4_mm_day * days_in_month

annual_mean_f4= field4.resample(time='YS').mean()
clim_f4 = annual_mean_f4.mean(dim='time')


ds4_SH_raw=xr.open_dataset('/bettik/PROJECTS/pr-regional-climate/santolam/gmted/'+'gmted_on_aphro.nc')['elevation']
ds4_SH=ds4_SH_raw.rename({'latitude':'lat','longitude':'lon'})
ds4_SH=dom.reversing_lat(ds4_SH)
field4_SH=ds4_SH.sel(lat=slice(latN,latS),lon=slice(lonW,lonE))



In [ ]:
#clim_f4.plot()

In [ ]:
#print(clim_f4.max(),clim_f4.min())

In [ ]:
##ax5
ds_RF_raw=xr.open_dataset(f'/bettik/PROJECTS/pr-regional-climate/santolam/MARout_coarser/'+
                            f'RF_mon_MARv3.14_ER5_spin2_{domainM}_1950-2025_aphro025x025.nc')['RF']
mask = np.isfinite(ds_RF_raw).any(dim='TIME')  # cells valid at least once
ds_RF = ds_RF_raw.where(mask, drop=True)
ds_SF_raw=xr.open_dataset(f'/bettik/PROJECTS/pr-regional-climate/santolam/MARout_coarser/'+
                            f'SF_mon_MARv3.14_ER5_spin2_'+domainM+'_1950-2025_aphro025x025.nc')['SF']
maskSF = np.isfinite(ds_SF_raw).any(dim='TIME')  # cells valid at least once
ds_SF = ds_SF_raw.where(maskSF, drop=True)
ds_TF=ds_RF+ds_SF

ds5 = ds_TF.rename({'TIME': 'time','x':'longitude','y':'latitude'})
field5=ds5.sel(time=slice(str(iyr), str(fyr))).sel(latitude=slice(latS,latN),longitude=slice(lonW,lonE))

field5=dom.reversing_lat(field5)

annual_mean_f5 = field5.resample(time='YS').mean()
clim_f5=annual_mean_f5.mean(dim='time')

bias_f5=clim_f5.values-clim_f4.values


In [ ]:
#bias_f5.shape

In [ ]:
#plt.pcolormesh(bias_f5)
#plt.colorbar()

In [ ]:
##ax6
iyr=2000
fyr=2016

sourceData1=f'/bettik/PROJECTS/pr-regional-climate/santolam/MARout_post/{domainM}/spin2/work/snow_bin/'
ds1_raw=xr.open_mfdataset(sourceData1+f'SHSN3_bin_daymean_MARv3.14_ER5_spin2_{domainM}_*.nc')['SHSN3']
ds1=ds1_raw[:,ny:-ny,nx:-nx].sel(TIME=slice(str(iyr), str(fyr)))

ds1_snow_days_per_year = ds1.resample(TIME="Y").sum()
#ds1_mean_snow_days = ds1_snow_days_per_year.mean(dim="time")

ds1_days_per_year = ds1.resample(TIME="Y").count(dim='TIME')
ds1_snow_days_percentage = (ds1_snow_days_per_year / ds1_days_per_year)# * 100
clim_f6=100*ds1_snow_days_percentage.mean('TIME')

In [ ]:
domain='HK' #CH
model3='hmasr'
sourceData3='/bettik/PROJECTS/pr-regional-climate/santolam/HMA_SR_daily_7km_'+domain+'/'
ds3_raw=xr.open_mfdataset(sourceData3+'/snow_bin/'+
                f'snd_bin_HMA_SR_D_v01_WY*_*_{domain}_lat35N43N_lon66E79E_SD_POST_MASKED_coarsen15x15.nc')['SD_Post']
            #f'snd_bin_HMA_SR_D_v01_WY*_*_{domain}_lat24N32N_lon80E91E_SD_POST_MASKED_coarsen15x15.nc')['SD_Post']

ds3=ds3_raw.sel(lat=slice(latS,latN),lon=slice(lonW,lonE))
ds3_snow_days_per_year = ds3.resample(time="Y").sum()
#ds2_mean_snow_days = ds2_snow_days_per_year.mean(dim="time")
ds3_days_per_year = ds3.resample(time="Y").count(dim='time')
ds3_snow_days_percentage = (ds3_snow_days_per_year / ds3_days_per_year) #* 100
clim_f7=100*ds3_snow_days_percentage.mean('time')

In [ ]:
#ds3_raw

In [ ]:
##
ds_grid_model=ds_grM
src_lons = ds_grid_model['LON'][ny:-ny, nx:-nx]
src_lats = ds_grid_model['LAT'][ny:-ny, nx:-nx]

tgt_lons = ds3['lon']
tgt_lats = ds3['lat']
ds1_geo = ds1.assign_coords(
    lon=(('Y', 'X'), src_lons.data),
    lat=(('Y', 'X'), src_lats.data)
)
import xesmf as xe

# Ensure lat/lon coordinates are attached properly
ds1_geo = ds1.to_dataset(name='SHSN3').assign_coords(
    lon=(('Y', 'X'), src_lons.data),
    lat=(('Y', 'X'), src_lats.data)
)

# Make sure ds3 has 'lat' and 'lon' (2D if needed)
if ds3['lat'].ndim == 1 and ds3['lon'].ndim == 1:
    ds3_grid = xr.Dataset({
        'lat': (['lat'], ds3['lat'].data),
        'lon': (['lon'], ds3['lon'].data)
    })
else:
    ds3_grid = xr.Dataset({'lat': ds3['lat'], 'lon': ds3['lon']})

# Create the regridder
regridder = xe.Regridder(ds1_geo, ds3_grid, method='bilinear', reuse_weights=False)

# Apply interpolation
ds1_interp = regridder(ds1_geo)
ds1_interp=ds1_interp.SHSN3
ds1_interp_snow_days_per_year = ds1_interp.resample(TIME="Y").sum()
#ds1_mean_snow_days = ds1_snow_days_per_year.mean(dim="time")

ds1_interp_days_per_year = ds1_interp.resample(TIME="Y").count(dim='TIME')
ds1_interp_snow_days_percentage = (ds1_interp_snow_days_per_year / ds1_interp_days_per_year)# * 100

clim_f8=100*ds1_interp_snow_days_percentage.mean('TIME')

bias_f8=clim_f8.values-clim_f7.values

In [ ]:
ds7_SH_raw=xr.open_dataset('/bettik/PROJECTS/pr-regional-climate/santolam/gmted/'+'gmted_on_hma_sr_HK_7km.nc')['elevation']
#ds7_SH=ds7_SH_raw.rename({'latitude':'lat','longitude':'lon'})
#ds7_SH=dom.reversing_lat(ds7_SH)
field7_SH=ds7_SH_raw.sel(lat=slice(latS,latN),lon=slice(lonW,lonE))
#field7_SH

In [ ]:
###HMA SR
domain='HK' #CH
model7='hmasr'
sourceData7='/bettik/PROJECTS/pr-regional-climate/santolam/HMA_SR_daily_7km_'+domain+'/'
#ds3_mask_raw=xr.open_mfdataset(sourceData7+"/*/"+'HMA_SR_D_v01_WY20*_HK_lat35N43N_lon66E79E_MASK_coarsen15x15.nc', combine="nested", concat_dim="time")#['Classification_mask']
ds3_mask_raw=xr.open_dataset(sourceData7+'/2016/'+
                    'HMA_SR_D_v01_N42_0E71_0_agg_16_WY2016_17_SD_POST_MASKED_mostcomplete_coarsen15x15.nc')#['Non_seasonal_snow_mask']
ds3_mask=ds3_mask_raw['Non_seasonal_snow_mask']
#ds3_mask=dom.reversing_lat(ds3_mask_lat)
#ds3_mask[:,:].plot()

In [ ]:
###MASKS

dsICE_HMASR_raw=100*ds3_mask.sel(Latitude=slice(latN,latS),Longitude=slice(lonW,lonE))
dsICE_HMASR_raw= dsICE_HMASR_raw.rename({
    'Latitude': 'lat',
    'Longitude': 'lon'
})

dsICE_HMASR=dsICE_HMASR_raw.transpose('lat', 'lon')
#dsICE_HMASR.plot()
print(dsICE_HMASR.min(),dsICE_HMASR.max())

In [ ]:
##
#MASK MAR
#dsICE.plot()
print(dsICE.min(),dsICE.max())

In [ ]:
###I interpolate MAR ICE to HMASR grid
##
#dsICE interpolation
##
ds_grid_model=ds_grM
src_lons = ds_grid_model['LON'][ny:-ny, nx:-nx]
src_lats = ds_grid_model['LAT'][ny:-ny, nx:-nx]

tgt_lons = ds3['lon']
tgt_lats = ds3['lat']
ds1_ICE_geo = dsICE.assign_coords(
    lon=(('y', 'x'), src_lons.data),
    lat=(('y', 'x'), src_lats.data)
)
import xesmf as xe

# Ensure lat/lon coordinates are attached properly
ds1_ICE_geo= dsICE.to_dataset(name='ICE').assign_coords(
    lon=(('y', 'x'), src_lons.data),
    lat=(('y', 'x'), src_lats.data)
)

# Make sure ds3 has 'lat' and 'lon' (2D if needed)
if ds3['lat'].ndim == 1 and ds3['lon'].ndim == 1:
    ds3_grid = xr.Dataset({
        'lat': (['lat'], ds3['lat'].data),
        'lon': (['lon'], ds3['lon'].data)
    })
else:
    ds3_grid = xr.Dataset({'lat': ds3['lat'], 'lon': ds3['lon']})

# Create the regridder
regridder = xe.Regridder(ds1_ICE_geo, ds3_grid, method='bilinear', reuse_weights=False)

# Apply interpolation
dsICE_interp = regridder(ds1_ICE_geo)

In [ ]:
dsICE_interp=dsICE_interp.ICE

In [ ]:
#MASK MAR
#dsICE_interp.plot()
#print(dsICE_interp.min(),dsICE_interp.max())

In [ ]:
##Original MAR
mask1_MAR_more70 = dsICE > 50
mask1_MAR_less10 = dsICE < 10

##Original HMASR
mask1_HMASR_more70 = dsICE_HMASR > 50
mask1_HMASR_less10 = dsICE_HMASR <10
##Interpolated MAR to HMASR

##Interpolated MAR
mask1_MAR_interp_more70 = dsICE_interp > 50
mask1_MAR_interp_less10 = dsICE_interp <10


##OR
combined_mask_less10=mask1_HMASR_less10 | mask1_MAR_interp_less10
combined_mask_more70=mask1_HMASR_more70 | mask1_MAR_interp_more70

In [ ]:
#combined_mask_more70

In [ ]:
std_f1=annual_mean_f1.std(dim='time')
std_f4=annual_mean_f4.std(dim='time')
std_f8=100*ds1_interp_snow_days_percentage.std('TIME')

In [ ]:
clevs3 = np.arange(5, 100, 10)
cmap_mod3 = plt.get_cmap('Blues', len(clevs3) + 1)
colors3 = [cmap_mod3(j) for j in range(cmap_mod3.N)]
colors3[0] = (1, 1, 1, 1)  # white for 0
cmap3 = mcolors.ListedColormap(colors3)
norm3 = mcolors.BoundaryNorm(clevs3, cmap3.N, extend='both')

clevs6 = np.arange(5, 105, 5)
cmap_mod6 = plt.get_cmap('YlGnBu', len(clevs6) + 1)
colors6 = [cmap_mod6(j) for j in range(cmap_mod6.N)]
colors6[0] = (1, 1, 1, 1)  # white for 0
cmap6 = mcolors.ListedColormap(colors6)
norm6 = mcolors.BoundaryNorm(clevs6, cmap6.N, extend='both')

clevsList=[
np.arange(-30,33,3),np.arange(-30,33,3),np.arange(-14, 16, 2),
clevs3, clevs3,np.arange(-50, 55, 5),
clevs6,clevs6,np.arange(-50, 55, 5)
]
cmapList=[
plt.get_cmap('RdYlBu_r'),plt.get_cmap('RdYlBu_r'),plt.get_cmap('bwr'),
cmap3 ,cmap3 ,plt.get_cmap('BrBG'),
cmap6 ,cmap6 ,  plt.get_cmap('bwr_r')  
]


norm0=mcolors.BoundaryNorm(clevsList[0], cmapList[0].N, extend='both')
norm2=mcolors.BoundaryNorm(clevsList[2], cmapList[2].N, extend='both')
norm5=mcolors.BoundaryNorm(clevsList[5], cmapList[5].N, extend='both')
norm8=mcolors.BoundaryNorm(clevsList[8], cmapList[8].N, extend='both')

normList=[
    norm0,norm0,norm2,
    norm3,norm3,norm5,
    norm6,norm6,norm8
]


In [ ]:
fig, axs = plt.subplots(
    3, 3, figsize=(28, 28),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

ax0 = axs[0,0];ax1 = axs[0,1];ax2 = axs[0,2]

ax3 = axs[1,0];ax4 = axs[1,1];ax5 = axs[1,2]

ax6 = axs[2,0];ax7 = axs[2,1];ax8 = axs[2,2]

##Temperature
lons0,lats0=LONS,LATS
im0 = ax0.pcolormesh(lons0, lats0, clim_f0,
                     transform=ccrs.PlateCarree(),
                     cmap=cmapList[0], norm=normList[0], shading='auto')
ax0.add_feature(cfeature.COASTLINE)
ax0.add_feature(cfeature.BORDERS, linestyle=':')
#ax0.set_title("(a1) Temperature – Model", fontsize=18)
SH0 = ax0.contour(lons0, lats0, field0_SH,
                     levels=[1000,2000,3000,4000,5000,6000,7000],
                     colors="k", linewidths=0.5, alpha=0.7,
                     transform=ccrs.PlateCarree())
ax0.clabel(SH0, inline=True, fontsize=10, fmt="%d")
#ICE1 = ax0.contourf(lons0, lats0, field0_ICE,
##                           levels=[10,50,70,100],
 #                          hatches=['','X','XX'],
 #                          colors="none",
 #                          transform=ccrs.PlateCarree())
ax0.set_aspect(1.4)
# Obs
lons1,lats1=clim_f1.lon,clim_f1.lat
im1 = ax1.pcolormesh(lons1, lats1, clim_f1,
                     transform=ccrs.PlateCarree(),
                     cmap=cmapList[1], norm=normList[1], shading='auto')
ax1.add_feature(cfeature.COASTLINE)
ax1.add_feature(cfeature.BORDERS, linestyle=':')
SH1 = ax1.contour(lons1, lats1, field1_SH,
                     levels=[1000,2000,3000,4000,5000,6000,7000],
                     colors="k", linewidths=0.5, alpha=0.7,
                     transform=ccrs.PlateCarree())
ax1.clabel(SH1, inline=True, fontsize=10, fmt="%d")

#ax1.set_title("(a2) Temperature – Obs", fontsize=18)
ax1.set_aspect(1.4)
# Bias
im2 = ax2.pcolormesh(lons1, lats1, bias_f2,
                     transform=ccrs.PlateCarree(),
                     cmap=cmapList[2], norm=normList[2], shading='auto')
ax2.add_feature(cfeature.COASTLINE)
ax2.add_feature(cfeature.BORDERS, linestyle=':')
SH1 = ax2.contour(lons1, lats1, field1_SH,
                     levels=[1000,2000,3000,4000,5000,6000,7000],
                     colors="k", linewidths=0.5, alpha=0.7,
                     transform=ccrs.PlateCarree())
sig_T_f2 = np.abs(bias_f2) > std_f1
mpl.rcParams['hatch.color'] ='black' #colorice'chartreuse'
mpl.rcParams['hatch.linewidth'] =2  # optional


ax2.contourf(
    lons1, lats1,
    sig_T_f2,
    levels=[0,0.5],
    colors='none',
    hatches=['.'],
    transform=ccrs.PlateCarree(),
    zorder=8
)
ax2.clabel(SH1, inline=True, fontsize=10, fmt="%d")
ax2.set_aspect(1.4)
#ax2.set_title("(a3) Temperature – Bias", fontsize=18)

# -----------------------------------------
#Precip

# Model
im3 = ax3.pcolormesh(lons0, lats0, clim_f3,
                     transform=ccrs.PlateCarree(),
                     cmap=cmapList[3], norm=normList[3], shading='auto')
ax3.add_feature(cfeature.COASTLINE)
ax3.add_feature(cfeature.BORDERS, linestyle=':')
#ax3.set_title("(b1) Precip – Model", fontsize=18)
SH0 = ax3.contour(lons0, lats0, field0_SH,
                     levels=[1000,2000,3000,4000,5000,6000,7000],
                     colors="k", linewidths=0.5, alpha=0.7,
                     transform=ccrs.PlateCarree())
ax3.clabel(SH0, inline=True, fontsize=10, fmt="%d")
ax3.set_aspect(1.4)
# Obs
lons4,lats4=clim_f4.lon,clim_f4.lat
im4 = ax4.pcolormesh(lons4, lats4, clim_f4,
                     transform=ccrs.PlateCarree(),
                     cmap=cmapList[4], norm=normList[4], shading='auto')
ax4.add_feature(cfeature.COASTLINE)
ax4.add_feature(cfeature.BORDERS, linestyle=':')
SH4 = ax4.contour(lons4, lats4, field4_SH,
                     levels=[1000,2000,3000,4000,5000,6000,7000],
                     colors="k", linewidths=0.5, alpha=0.7,
                     transform=ccrs.PlateCarree())
ax4.clabel(SH4, inline=True, fontsize=10, fmt="%d")
ax4.set_aspect(1.4)
#ax4.set_title("(b2) Precip – Obs", fontsize=18)

# Bias
im5 = ax5.pcolormesh(lons4, lats4, bias_f5,
                     transform=ccrs.PlateCarree(),
                     cmap=cmapList[5], norm=normList[5], shading='auto')
ax5.add_feature(cfeature.COASTLINE)
ax5.add_feature(cfeature.BORDERS, linestyle=':')
SH5 = ax5.contour(lons4, lats4, field4_SH,
                     levels=[1000,2000,3000,4000,5000,6000,7000],
                     colors="k", linewidths=0.5, alpha=0.7,
                     transform=ccrs.PlateCarree())
sig_P_f5 = np.abs(bias_f5) > std_f4
mpl.rcParams['hatch.color'] ='black' #colorice'chartreuse'
mpl.rcParams['hatch.linewidth'] =2  # optional

ax5.contourf(
    lons4, lats4,
    sig_P_f5,
    levels=[0,0.5],
    colors='none',
    hatches=['.'],
    transform=ccrs.PlateCarree(),
    zorder=8
)
ax5.clabel(SH5, inline=True, fontsize=10, fmt="%d")
ax5.set_aspect(1.4)
#ax5.set_title("(b3) Precip – Bias", fontsize=18)

# -----------------------------------------
# SNOW ROW
# -----------------------------------------
colorice='orange'
# Model Original
im6 = ax6.pcolormesh(lons0, lats0, clim_f6,
                     transform=ccrs.PlateCarree(),
                     cmap=cmapList[6], norm=normList[6], shading='auto')

ax6.add_feature(cfeature.COASTLINE)
ax6.add_feature(cfeature.BORDERS, linestyle=':')
SH0 = ax6.contour(lons0, lats0, field0_SH,
                     levels=[1000,2000,3000,4000,5000,6000,7000],
                     colors="k", linewidths=0.5, alpha=0.7,
                     transform=ccrs.PlateCarree())
ax6.clabel(SH0, inline=True, fontsize=10, fmt="%d")
#ax6.set_title("(c1) Snow – Model", fontsize=18)

#cs6_10= ax6.contour(lons0,lats0,mask1_MAR_less10,levels=[0.5],colors=["black"],linewidths=2,transform=ccrs.PlateCarree())
cs6_70= ax6.contour(lons0,lats0,mask1_MAR_more70,levels=[0.5],colors=colorice,linewidths=2,transform=ccrs.PlateCarree())
mpl.rcParams['hatch.color'] =colorice #colorice'chartreuse'
mpl.rcParams['hatch.linewidth'] =2  # optional

ax6.contourf(
    lons0,lats0,mask1_MAR_more70,
    levels=[0.5, 1],
    colors='white',          # better than white if you only want hatching
    hatches=['///'],
    transform=ccrs.PlateCarree()
)
#ax6.clabel(cs6_10, fmt={0.5: '10'}, fontsize=12)
#ax6.clabel(cs6_70, fmt={0.5: '10'}, fontsize=12)

ax6.set_aspect(1.5)

# Obs
lons7,lats7=clim_f7.lon,clim_f7.lat
im7 = ax7.pcolormesh(lons7, lats7, clim_f7,
                     transform=ccrs.PlateCarree(),
                     cmap=cmapList[7], norm=normList[7], shading='auto')
ax7.add_feature(cfeature.COASTLINE)
ax7.add_feature(cfeature.BORDERS, linestyle=':')
SH7 = ax7.contour(lons7, lats7, field7_SH,
                     levels=[1000,2000,3000,4000,5000,6000,7000],
                     colors="k", linewidths=0.5, alpha=0.7,
                     transform=ccrs.PlateCarree())
ax7.clabel(SH7, inline=True, fontsize=10, fmt="%d")

lons7_m,lats7_m=mask1_HMASR_less10.lon,mask1_HMASR_less10.lat
#cs7_10= ax7.contour(lons7_m,lats7_m,mask1_HMASR_less10,levels=[0.5],colors=["black"],linewidths=2,transform=ccrs.PlateCarree())
cs7_70= ax7.contour(lons7_m,lats7_m,mask1_HMASR_more70,levels=[0.5],colors=colorice,linewidths=2,transform=ccrs.PlateCarree())

mpl.rcParams['hatch.color'] =colorice #colorice'chartreuse'
mpl.rcParams['hatch.linewidth'] =2  # optional

ax7.contourf(
    lons7_m,lats7_m,mask1_HMASR_more70,
    levels=[0.5, 1],
    colors='white',          # better than white if you only want hatching
    hatches=['///'],
    transform=ccrs.PlateCarree()
)
#ax7.clabel(cs7_10, fmt={0.5: '10'}, fontsize=12)
#ax7.clabel(cs7_70, fmt={0.5: '70'}, fontsize=12)

ax7.set_aspect(1.5)

#########
im8 = ax8.pcolormesh(
    lons7,lats7,bias_f8,
    transform=ccrs.PlateCarree(),
    cmap=cmapList[8], norm=normList[8],
    shading='auto'
)

ax8.add_feature(cfeature.COASTLINE)
ax8.add_feature(cfeature.BORDERS, linestyle=':')
SH8 = ax8.contour(field7_SH.lon, field7_SH.lat, field7_SH,
                     levels=[1000,2000,3000,4000,5000,6000,7000],
                     colors="k", linewidths=0.5, alpha=0.7,
                     transform=ccrs.PlateCarree())
ax8.clabel(SH8, inline=True, fontsize=10, fmt="%d")

lons8,lats8=combined_mask_more70.lon,combined_mask_more70.lat
#cs8_10= ax8.contour(lons8,lats8,combined_mask_less10,levels=[0.5],colors=["black"],linewidths=2,transform=ccrs.PlateCarree())
import matplotlib as mpl

# Set hatch color
mpl.rcParams['hatch.color'] =colorice #colorice'chartreuse'
mpl.rcParams['hatch.linewidth'] =2  # optional

cs8_70 = ax8.contour(
    lons8, lats8,
    combined_mask_more70,
    levels=[0.5],
    colors=colorice,
    linewidths=2,
    transform=ccrs.PlateCarree()
)

ax8.contourf(
    lons8, lats8,
    combined_mask_more70,
    levels=[0.5, 1],
    colors='white',          # better than white if you only want hatching
    hatches=['///'],
    transform=ccrs.PlateCarree()
)
sig_S_f8 = np.abs(bias_f8) > std_f8
sig_S_f8 = sig_S_f8.where(np.abs(bias_f8) >= 1)

mpl.rcParams['hatch.color'] ='black' #colorice'chartreuse'
mpl.rcParams['hatch.linewidth'] =2  # optional

ax8.contourf(
    lons7, lats7,
    sig_S_f8,
    levels=[0, 0.5],
    colors='none',
    hatches=['.'],
    transform=ccrs.PlateCarree(),
    zorder=1
)
#ax8.clabel(cs8_10, fmt={0.5: '10'}, fontsize=12)
#ax8.clabel(cs8_70, fmt={0.5: '70'}, fontsize=12)
ax8.set_aspect(1.5)






#############################################
# Zero-isoline for temperature row
ax0.contour(lons0, lats0, clim_f0, levels=[0], colors="blue", linewidths=2)
ax1.contour(lons1, lats1, clim_f1, levels=[0], colors="blue", linewidths=2)
#ax2.contour(lons, lats, clim_f0, levels=[0], colors="blue", linewidths=2)


# -----------------------------------------
#  GRIDLINES 
# -----------------------------------------
axs_f = axs.flatten()

for i, ax in enumerate(axs_f):

    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=1,
        color="gray",
        alpha=0.3,
        linestyle=":"
    )

    gl.xlabel_style = {"size": 20}
    gl.ylabel_style = {"size": 20}

    # Remove labels on all sides first
    gl.top_labels = False
    gl.bottom_labels = False
    gl.left_labels = False
    gl.right_labels = False

    # Apply desired labeling logic
    if i in [0, 3, 6]:           # left column
        gl.top_labels = True
        gl.bottom_labels = True
        gl.left_labels = True

    elif i in [1, 4, 7]:         # middle column
        gl.top_labels = True
        gl.bottom_labels = True

    elif i in [2, 5, 8]:         # right column
        gl.top_labels = True
        gl.bottom_labels = True
        gl.right_labels = True
        
###
## Letters
###
letterList = ["a)MAR", "b)CRU", "c)$\Delta$ [MAR-CRU]", "d)MAR", "e)APHRO", "f)$\Delta$ [MAR-APHRO]", "g)MAR",
              "h)HMA SR", "i)$\Delta$ [MAR-HMA SR]"]
for i, ax in enumerate(axs.flatten()):
    ax.text(
        0.005, 0.99,            # position inside axes (left, top)
        letterList[i],
        transform=ax.transAxes,
        fontsize=24,
        fontweight='bold',
        va='top',
        ha='left',
        bbox=dict(facecolor='white', edgecolor='none', alpha=1.0, pad=2)
    )
# -----------------------------------------
# COLORBARS
# -----------------------------------------
cbar_loc_ax0= fig.add_axes([0.20, 0.66, 0.3, 0.02]) ##[x0, y0, width, height]
cbar_ax0 = fig.colorbar(im0, cax=cbar_loc_ax0, orientation="horizontal")
cbar_ax0.set_label("[°C]", fontsize=20)
cbar_ax0.ax.tick_params(labelsize=20)

cbar_loc_ax2= fig.add_axes([0.65, 0.66, 0.3, 0.02])
cbar_ax2 = fig.colorbar(im2, cax=cbar_loc_ax2, orientation="horizontal")
cbar_ax2.set_label("$\Delta$T [°C]", fontsize=20)
cbar_ax2.ax.tick_params(labelsize=20)

cbar_loc_ax3= fig.add_axes([0.20, 0.34, 0.3, 0.02]) ##[x0, y0, width, height]
cbar_ax3 = fig.colorbar(im3, cax=cbar_loc_ax3, orientation="horizontal")
cbar_ax3.set_label(label="[mm/month]",fontsize=20)
cbar_ax3.ax.tick_params(labelsize=20)

cbar_loc_ax5= fig.add_axes([0.65, 0.34, 0.3, 0.02])
cbar_ax5 = fig.colorbar(im5, cax=cbar_loc_ax5, orientation="horizontal")
cbar_ax5.set_label(label="$\Delta$ PRE[mm/month]",fontsize=20)
cbar_ax5.ax.tick_params(labelsize=20)

cbar_loc_ax6= fig.add_axes([0.20, 0.02, 0.3, 0.02]) ##[x0, y0, width, height]
cbar_ax6 = fig.colorbar(im6, cax=cbar_loc_ax6, orientation="horizontal")
cbar_ax6.set_label(label="[%]",fontsize=20)
cbar_ax6.ax.tick_params(labelsize=20)

cbar_loc_ax8= fig.add_axes([0.65, 0.02, 0.3, 0.02])
cbar_ax8  = fig.colorbar(im8, cax=cbar_loc_ax8, orientation="horizontal")
cbar_ax8.set_label(label="$\Delta$SCD [%]",fontsize=20)
cbar_ax8.ax.tick_params(labelsize=20)

# -----------------------------------------
# Layout + Save
# -----------------------------------------
row_labels = ["Temperature", "Precipitation", "Snow cover duration"]

for i, label in enumerate(row_labels):
    fig.text(
        -0.016,                                   # x-position (outside left margin)
        0.82 - i * (1/3),                        # y-positions for each row
        label,
        va='center',
        ha='center',
        rotation='vertical',
        fontsize=28#,
       # fontweight='bold'
    )

plt.subplots_adjust(left=0.03, right=0.95, top=0.95, bottom=0.05,
                    wspace=0.1, hspace=0.25)

#plotsDir="/home/santolam/MyPythonLibrary/MAR-RECAP/scripts_paper_on_evaluation_MAR/figures_paper_on_evaluation/"
#plotname=f"FIG2_clim_bias_pamir_icemask_clean_{colorice}_hatchedall_statsig"
#plt.savefig(f"{plotsDir}{plotname}.png", bbox_inches="tight")
#plt.savefig(plotsDir + plotname + '.eps', format='eps', dpi=300)
plt.show()